# **Counterfactuals for text, images and time series: Practice**

**Hi, friends.**

Tamara got the job after all: she is now an ML engineer and explains model decisions with counterfactuals herself. In the lesson “Extending across modalities: MiCE, Polyjuice, DiME, Native-Guide” we saw that the statement “validity + proximity + plausibility” carries over from tables to text, images and time series, and only the “minimal change” operation and the plausibility model change. Here we check this in code on three open datasets.

In this practice you will:
- write the word-level Levenshtein distance that MiCE uses to measure the minimality of an edit;
- build MiCE on DistilBERT SST-2 and `t5-small`: fine-tune the editor to restore words given a label, then find by binary search the smallest gradient-based mask that flips the prediction;
- generate review variants with the Polyjuice model, filter them by GPT-2 fluency and by the classifier's prediction, and compare them with MiCE;
- find region replacements in the spirit of Goyal et al. on the ResNet-50 feature map for the pair “striped cat and Egyptian cat” and compare the greedy choice with a random one;
- train the FCN series classifier on GunPoint and build Native-Guide: the closest series of the other class, CAM along time and the shortest rewritten segment.

The tabular methods of the module are collected in the notebook `PRACTICE_recourse`; this one covers only text, images and time series.

**Happy coding!**

### Which libraries we use and why not the others

| Method | Authors' implementation | On Colab |
|---|---|---|
| Polyjuice | package `polyjuice-nlp` 0.1.3, model `uw-hai/polyjuice` | the 0.1.3 wheel on PyPI installs without the `polyjuice` module (it holds metadata only), so we take the model from Hugging Face through `transformers` and the prompt format from the authors' code |
| MiCE | repository `allenai/mice` | not published as a package and built on AllenNLP, which is no longer developed; we write the editor and the binary search ourselves with `transformers` |
| Region replacements (Goyal et al.) | code accompanying the paper | no package; we write the greedy search in torch |
| Native-Guide | repository `e-delaney/Instance-Based_CFE_TSC` | written with TensorFlow and tslearn; we write the classifier and the search in torch |
| DiME | repository `guillaumejs2403/DiME` | needs a diffusion model trained on face images or ImageNet and hundreds of diffusion steps per image; we do not compute it in this practice, it stays in the theory of the lesson |

All models and data are open and download without keys: DistilBERT SST-2, `t5-small`, `uw-hai/polyjuice` and GPT-2 from Hugging Face, SST-2 reviews through `datasets`, ResNet-50 from `torchvision`, photos from the `data` folder of the course repository, GunPoint series from the UCR/UEA archive. On a Colab CPU the notebook takes about 10 minutes, most of it fine-tuning the MiCE editor and Polyjuice generation; a GPU is faster. The numbers in the sections with training and generation may differ slightly on other hardware.

In [ ]:
!pip install transformers==5.18.0 datasets==4.8.5 sentencepiece==0.2.2 -q   # already in Colab; we pin the versions of the run

In [ ]:
import time, io, zipfile, urllib.request, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torchvision
import torchvision.transforms.functional as TF
from PIL import Image
from IPython.display import display
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForSeq2SeqLM, AutoModelForCausalLM)

device = "cuda" if torch.cuda.is_available() else "cpu"
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all(0)
pd.set_option("display.max_colwidth", 120)
print("device:", device)

## 1. Text: the classifier and the reviews

The classifier is DistilBERT fine-tuned on SST-2 (binary sentiment of movie reviews). From the validation part of SST-2 we take the first 12 reviews of 6 to 20 words on which the classifier is correct: these are the predictions we will flip.

In [ ]:
CLF = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
ctok = AutoTokenizer.from_pretrained(CLF)
clf = AutoModelForSequenceClassification.from_pretrained(CLF).to(device).eval()
LAB = ["negative", "positive"]

def probs(texts):                                  # SST-2 class probabilities for a list of texts
    with torch.no_grad():
        e = ctok(texts, return_tensors="pt", padding=True, truncation=True).to(device)
        return clf(**e).logits.softmax(-1).cpu().numpy()

val = load_dataset("stanfordnlp/sst2", split="validation")
cand = [r for r in val if 6 <= len(r["sentence"].split()) <= 20]
P0 = probs([r["sentence"] for r in cand])
texts = [r["sentence"].strip() for r, pr in zip(cand, P0) if pr.argmax() == r["label"]][:12]
y0 = probs(texts).argmax(1)
pd.DataFrame({"text": texts, "prediction": [LAB[k] for k in y0]})

### Word-level Levenshtein distance

MiCE measures the minimality of an edit by the word-level Levenshtein distance (the box in the step “MiCE — minimal contrastive edits” of the lesson): the minimal number of word insertions, deletions and substitutions that turn $x$ into $x'$,

$$\mathrm{lev}(i,j)=\min\big\{\mathrm{lev}(i-1,j)+1,\ \mathrm{lev}(i,j-1)+1,\ \mathrm{lev}(i-1,j-1)+\mathbb{1}[x_i\neq x'_j]\big\},$$

with the base $\mathrm{lev}(i,0)=i$, $\mathrm{lev}(0,j)=j$.

**Task 1.** Complete `lev(a, b)` with a row-by-row dynamic program: keep one row `D` of the table, update it by the recurrence for every word $a_i$ and return its last element.

*Self-check question:* what is the distance for the pair from the lesson's task, “assembled a portfolio of three mock-ups” and “assembled a large portfolio of five mock-ups”?

<details><summary>Check yourself</summary>

$\mathrm{lev} = 2$: one insertion (“large”) and one substitution (“three” by “five”). Solution: `D = np.arange(len(b) + 1)`, then for every `i` keep `prev = D.copy()`, set `D[0] = i` and for every `j` take `min(prev[j] + 1, D[j - 1] + 1, prev[j - 1] + (a[i - 1] != b[j - 1]))`.
</details>

In [ ]:
def lev(a, b):                                     # word-level Levenshtein distance
    a, b = a.split(), b.split()
    ...  # Your code here: the row D of the dynamic program and the recurrence over words

print(lev("assembled a portfolio of three mock-ups", "assembled a large portfolio of five mock-ups"))

## 2. MiCE: a minimal contrastive edit

MiCE (Ross, Marasović, Peters, 2021) solves the task in two stages (step “MiCE — minimal contrastive edits”).

**Stage 1.** A T5 editor learns to restore masked words given a label: the input is `label: <label>. input: <text with masks>`, the output is the content of the masks. The authors mask the words with the largest gradient and train T5-base; for speed we take `t5-small`, a random mask of 20 to 55% of words and 300 steps of 16 reviews from the training part of SST-2. On CPU this takes a couple of minutes; the result of training depends on hardware.

**Stage 2.** For a review with label $y$ and contrast label $y_c = 1 - y$ we compute the gradient norm of the $y_c$ loss over the embeddings of every word, mask the share $n$ of words with the largest norm, ask the editor to fill the mask given the label $y_c$ (beam search, 15 candidates) and check the candidates with the classifier. The share $n$ is found by binary search on $[0, 0.55]$ in 4 steps: if at least one candidate flips the prediction, the mask is reduced, otherwise it is increased. The answer is the candidate with the smallest $\mathrm{lev}$ at the smallest successful mask.

In [ ]:
ED = "google-t5/t5-small"
ttok = AutoTokenizer.from_pretrained(ED)
editor = AutoModelForSeq2SeqLM.from_pretrained(ED).to(device)

def mask_words(words, idx):                        # words with indices idx -> T5 sentinels, the target is their content
    src, tgt, k, prev = [], [], 0, False
    for i, w in enumerate(words):
        if i in idx:
            if not prev:
                src.append(f"<extra_id_{k}>"); tgt.append(f"<extra_id_{k}>"); k += 1
            tgt.append(w); prev = True
        else:
            src.append(w); prev = False
    tgt.append(f"<extra_id_{k}>")
    return " ".join(src), " ".join(tgt)

train = load_dataset("stanfordnlp/sst2", split="train")
train = train.filter(lambda r: 6 <= len(r["sentence"].split()) <= 30).shuffle(seed=0).select(range(4800))
g = np.random.default_rng(0)
opt = torch.optim.AdamW(editor.parameters(), 3e-4)
editor.train(); t0 = time.time()
for step in range(300):                            # stage 1: the editor learns to restore words given a label
    batch = train[step * 16:(step + 1) * 16]
    S, T = [], []
    for s, l in zip(batch["sentence"], batch["label"]):
        w = s.split(); k = max(1, int(len(w) * g.uniform(0.2, 0.55)))
        a, b = mask_words(w, set(g.choice(len(w), k, replace=False).tolist()))
        S.append(f"label: {LAB[l]}. input: {a}"); T.append(b)
    e = ttok(S, return_tensors="pt", padding=True).to(device)
    lab = ttok(T, return_tensors="pt", padding=True).input_ids.to(device)
    lab[lab == ttok.pad_token_id] = -100
    loss = editor(**e, labels=lab).loss
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0: print("step %d, loss %.2f, %.0f s" % (step, loss.item(), time.time() - t0))
editor.eval()
print("stage 1 in %.0f s" % (time.time() - t0))

**Task 2.** Complete the binary search step in `mice`: if some candidates flip the prediction (`ok` is not empty), store in `best` the pair (the candidate with the smallest `lev` to the original text, the mask share `mid`) and move the upper bound `hi` to `mid`; otherwise move the lower bound `lo` to `mid`.

*Self-check question:* on how many of the 12 reviews did MiCE flip the prediction, and what are the shortest edits?

<details><summary>Check yourself</summary>

In the solved copy MiCE flipped the prediction on 10 reviews of 12. The shortest edits change one word at mask 0.034: “charming” → “stale”, “slow” → “funny” at the end of the review “it 's slow -- very , very slow”, “taxes” → “best”. Fluency suffers in long edits: “the mesmerizing performances of the leads keep the film grounded…” became “the length of the leads of the film , and the length of the film” at mask 0.413. The numbers depend on fine-tuning the editor. Solution: `d = [lev(text, c) for c in ok]; best = (ok[int(np.argmin(d))], mid); hi = mid`, otherwise `lo = mid`.
</details>

In [ ]:
def word_grads(text, target):                      # gradient norm over embeddings, maximum over the word's tokens
    words = text.split()
    e = ctok(words, is_split_into_words=True, return_tensors="pt").to(device)
    emb = clf.distilbert.embeddings.word_embeddings(e.input_ids).detach().requires_grad_(True)
    out = clf(inputs_embeds=emb, attention_mask=e.attention_mask).logits
    (-out.log_softmax(-1)[0, target]).backward()   # gradient of the contrast-label loss
    gn = emb.grad[0].norm(dim=-1).cpu().numpy()
    s = np.zeros(len(words))
    for t, w in enumerate(e.word_ids()):
        if w is not None: s[w] = max(s[w], gn[t])
    return words, s

def infill(words, idx, yc, n_beams=15):            # T5 fills the mask given the contrast label, beam search
    src, _ = mask_words(words, idx)
    k = src.count("<extra_id_")
    ids = ttok(f"label: {LAB[yc]}. input: {src}", return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        gen = editor.generate(ids, num_beams=n_beams, num_return_sequences=n_beams, max_new_tokens=8 * k + 5)
    out = []
    for gseq in gen:
        dec = ttok.decode(gseq, skip_special_tokens=False).replace("<pad>", "").replace("</s>", "")
        txt = src
        for j in range(k):
            a = dec.find(f"<extra_id_{j}>"); b = dec.find(f"<extra_id_{j + 1}>")
            fill = dec[a + len(f"<extra_id_{j}>"): b if b >= 0 else None].strip() if a >= 0 else ""
            txt = txt.replace(f"<extra_id_{j}>", fill)
        out.append(" ".join(txt.split()))
    return out

def mice(text, steps=4, n_max=0.55):
    y = probs([text])[0].argmax(); yc = 1 - y
    words, s = word_grads(text, yc)
    order = np.argsort(-s)                         # words with the largest gradient first
    lo, hi, best, calls = 0.0, n_max, None, 0
    for _ in range(steps):                         # stage 2: binary search over the share of masked words
        mid = (lo + hi) / 2
        k = max(1, int(round(mid * len(words))))
        cands = infill(words, set(order[:k].tolist()), yc)
        pc = probs(cands)[:, yc]; calls += len(cands)
        ok = [c for c, q in zip(cands, pc) if q > 0.5]
        if ok:
        ...  # Your code here: update best and hi if ok is not empty, otherwise lo
    return y, best, calls

rows = []
t0 = time.time()
for t in texts:
    y, best, calls = mice(t)
    rows.append({"text": t, "was": LAB[y], "MiCE edit": best[0] if best else None,
                 "mask share": round(best[1], 3) if best else None,
                 "lev / n": round(lev(t, best[0]) / len(t.split()), 3) if best else None,
                 "p of contrast label": round(probs([best[0]])[0, 1 - y], 3) if best else None,
                 "calls": calls})
mice_df = pd.DataFrame(rows)
print("MiCE: flipped %d of %d, %.0f s" % (mice_df["MiCE edit"].notna().sum(), len(texts), time.time() - t0))
mice_df

MiCE has no separate fluency filter: fluency is held by the editor itself, and `t5-small` after 300 steps holds it poorly when the mask covers a third of the review. Where one or two words are enough, the edit reads as an explanation: “charming”, replaced by “stale”, is exactly the word that decided the prediction.

## 3. Polyjuice: fluent variants without looking at the classifier

Polyjuice (Wu, Ribeiro, Heer, Weld, 2021) is a fine-tuned GPT-2 that fills a gap given the original text, an edit code and the text with a `[BLANK]`. The prompt format from the authors' code is `<text> <|perturb|> [<code>] <text with [BLANK]> [SEP]`, and the answer ends with `[ANSWER]`. The generator does not see the classifier: candidates are generated first and filtered afterwards.

We put the blank on every word of the review in turn and try four codes for every blank: `negation`, `lexical`, `quantifier`, `resemantic`. The fluency filter is the one from the lesson's step: GPT-2 computes the total log-probability (natural logarithm) of the original text and of the variant, and the variant is dropped if it fell by more than 10; the authors also run the same check on the changed fragment, we limit ourselves to the whole sentence. Generation uses random sampling (`top_p = 0.9`) with a fixed seed.

In [ ]:
PJ = "uw-hai/polyjuice"
ptok = AutoTokenizer.from_pretrained(PJ)
pj = AutoModelForCausalLM.from_pretrained(PJ).to(device).eval()
gtok = AutoTokenizer.from_pretrained("openai-community/gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").to(device).eval()
CODES = ["negation", "lexical", "quantifier", "resemantic"]

def polyjuice(text, code, i):                      # prompt in the authors' format: text, edit code, text with [BLANK]
    w = text.split(); blanked = " ".join(w[:i] + ["[BLANK]"] + w[i + 1:])
    prompt = f"{text} <|perturb|> [{code}] {blanked} [SEP]"
    ids = ptok(prompt, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        out = pj.generate(ids, do_sample=True, top_p=0.9, max_new_tokens=12, pad_token_id=ptok.eos_token_id)
    fill = ptok.decode(out[0, ids.shape[1]:]).split("[ANSWER]")[0].replace("<|endoftext|>", "").strip()
    if fill == "EMPTY": fill = ""
    return " ".join(blanked.replace("[BLANK]", fill).split())

def logprob(texts):                                # total log-probability of a text under GPT-2 (natural logarithm)
    res = []
    for t in texts:
        ids = gtok(t, return_tensors="pt").input_ids.to(device)
        with torch.no_grad(): res.append(-gpt2(ids, labels=ids).loss.item() * (ids.shape[1] - 1))
    return np.array(res)

seed_all(0); t0 = time.time()
rows = []
for t in texts:
    y = probs([t])[0].argmax(); lp0 = logprob([t])[0]
    for i, w in enumerate(t.split()):
        if not any(ch.isalpha() for ch in w): continue    # punctuation is not replaced
        for code in CODES:
            v = polyjuice(t, code, i)
            if v == t: continue
            rows.append({"text": t, "was": y, "code": code, "variant": v, "lp0": lp0})
pj_df = pd.DataFrame(rows).drop_duplicates(["text", "variant"])
pj_df["Δ log-probability"] = logprob(pj_df["variant"].tolist()) - pj_df["lp0"]
pj_df["p of contrast label"] = [probs([v])[0, 1 - y] for v, y in zip(pj_df["variant"], pj_df["was"])]
pj_df["lev / n"] = [lev(t, v) / len(t.split()) for t, v in zip(pj_df["text"], pj_df["variant"])]
print("Polyjuice: %d variants in %.0f s" % (len(pj_df), time.time() - t0))

**Task 3.** Keep in `fluent` the fluent variants (Δ log-probability not below −10), and in `flipped` those of them whose probability of the contrast label is above 0.5.

*Self-check question:* how many variants passed the fluency filter, how many of them change the prediction, and on how many reviews did Polyjuice find at least one counterfactual?

<details><summary>Check yourself</summary>

In the solved copy 247 of 456 variants are fluent, 16 change the prediction, and a counterfactual was found for 7 reviews of 12. The codes `lexical` (8) and `negation` (5) change the prediction most often. The numbers depend on random generation. Solution: `fluent = pj_df[pj_df["Δ log-probability"] >= -10]`, `flipped = fluent[fluent["p of contrast label"] > 0.5]`.
</details>

In [ ]:
# Task 3: keep the fluent variants (Δ log-probability not below -10) and among them those that change the prediction
fluent = ...   # Your code here
flipped = ...  # Your code here
print("variants: %d, fluent: %d, of them change the prediction: %d" % (len(pj_df), len(fluent), len(flipped)))
print("texts where Polyjuice found at least one counterfactual: %d of %d" % (flipped["text"].nunique(), len(texts)))
display(flipped.groupby("code").size().rename("change prediction").to_frame()
        .join(fluent.groupby("code").size().rename("fluent")))
best_pj = flipped.sort_values("lev / n").groupby("text").head(1)
best_pj[["text", "code", "variant", "lev / n", "p of contrast label"]]

Let us put both methods side by side per review: did the method flip the prediction, and what share of words changed ($\mathrm{lev}/n$).

In [ ]:
cmp = pd.DataFrame({"text": texts,
                    "MiCE: lev / n": mice_df["lev / n"].values,
                    "Polyjuice: best lev / n": [best_pj.set_index("text")["lev / n"].get(t, np.nan) for t in texts]})
print("share of flipped texts: MiCE %.2f, Polyjuice %.2f"
      % (cmp["MiCE: lev / n"].notna().mean(), cmp["Polyjuice: best lev / n"].notna().mean()))
print("median lev / n where both found an edit: MiCE %.3f, Polyjuice %.3f"
      % tuple(cmp.dropna()[["MiCE: lev / n", "Polyjuice: best lev / n"]].median()))
cmp

In the solved copy MiCE flipped 10 reviews of 12 and Polyjuice 7 of 12, but where both found an edit, the Polyjuice edit is shorter: the median $\mathrm{lev}/n$ is 0.058 against 0.200. These are the two poles from the lesson's conclusion. MiCE is targeted: the gradient points at where to edit and the editor writes for the contrast label, so it finds an edit more often but pays with length and fluency. Polyjuice goes through one-word edits without looking at the model: the edits are short and fluent, but only a small share of them changes the prediction.

Among the edits Polyjuice found there are some that change the prediction without changing the meaning: “you do n't have to know about music to appreciate the film…” with “know” replaced by “care” is already a negative review for the classifier. Such a counterfactual explains not the review but a weakness of the classifier, and this is what Polyjuice is used for in model debugging.

## 4. Images: region replacements (Goyal et al.)

Goyal et al. (2019) take a query $I$ of class $c$ and a distractor $I'$ of the target class $c'$ and look on the feature map for the smallest number of cells whose replacement by the distractor's cells flips the decision (step “Counterfactuals on images: regions”):

$$\min_{a,\,P}\ \lVert a\rVert_1\quad\text{subject to}\quad c'=\arg\max\ g\big((1-a)\odot f(I) + a\odot P\,f(I')\big).$$

In ResNet-50 the map $f$ after the last block has $7 \times 7$ cells of 2048 features, and $g$ is the mean over cells and a linear layer. The mean is linear, so the logits after replacing query cell $i$ by distractor cell $j$ are computed for all $49 \cdot 49$ pairs at once, without running the network. The greedy search takes at every step the pair with the largest log-probability of $c'$; every cell of the query and of the distractor is used once, because $P$ is a permutation.

The photos come from the course's `data` folder. The query is `cat.jpg`, which ResNet-50 assigns to the class tiger cat (a striped domestic cat); the distractor is `cat_piran.jpg`, class Egyptian cat: close classes, like the pairs of bird species in the paper. The `IMAGENET1K_V2` weights were trained with label smoothing, so their top-class probabilities are low.

Cat photo: [Paul Hanaoka, Unsplash](https://unsplash.com/photos/w2DsS-ZAP4U), Unsplash License.

Photo of the cat from Piran (`cat_piran.jpg`): Jakub Hałun, [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Cat_in_Piran,_Slovenia,_20240504_1600_8594.jpg), license [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

Piglet photo: [Brett Sayles, Pexels](https://www.pexels.com/photo/4966539/), Pexels License.

Cat and pug photo: [Priscilla Du Preez, Unsplash](https://unsplash.com/photos/qtkpOxCpl0Y), Unsplash License.

Zebras photo (`geranimo-ADUi-P4n-Jwds-unsplash.jpg`): [Geranimo, Unsplash](https://unsplash.com/photos/ADUiP4nJwds), Unsplash License.

In [ ]:
DATA = "https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data"
def load_img(name): return Image.open(io.BytesIO(urllib.request.urlopen(f"{DATA}/{name}", timeout=30).read())).convert("RGB")

W50 = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
resnet = torchvision.models.resnet50(weights=W50).to(device).eval()
prep, CATS = W50.transforms(), W50.meta["categories"]
body = nn.Sequential(*list(resnet.children())[:-2])   # up to and including layer4: a 2048 x 7 x 7 map
Wfc, bfc = resnet.fc.weight.detach(), resnet.fc.bias.detach()

def cells(img):                                    # feature map as 49 cells of 2048 features
    with torch.no_grad(): return body(prep(img)[None].to(device))[0].flatten(1).T
def head(F): return F.mean(0) @ Wfc.T + bfc        # g: mean over cells and the linear layer fc

imgs = {n: load_img(n) for n in ["cat.jpg", "cat_piran.jpg", "geranimo-ADUi-P4n-Jwds-unsplash.jpg", "piglet.jpg", "cat_and_dog.jpg"]}
F = {n: cells(im) for n, im in imgs.items()}
pd.DataFrame({n: {"class": CATS[head(f).argmax().item()], "p": round(head(f).softmax(0).max().item(), 3)} for n, f in F.items()}).T

**Task 4.** Complete the greedy step in `goyal`: find the pair $(i, j)$ with the largest `s` (`s` is a $49 \times 49$ matrix, the pair index is `divmod(s.argmax().item(), 49)`), replace the cell `Fc[i]` by the cell `Fd[j]` and add the pair to `edits`.

*Self-check question:* how many replacements did it take to turn the striped cat into an Egyptian cat, and where are these cells in the photos?

<details><summary>Check yourself</summary>

Two replacements were enough: query cells 36 and 29 (rows 5 and 4, column 1, the table and the jug to the left of the cat, not the cat itself) by distractor cells 27 and 26 (row 3, columns 6 and 5, the gray cat's face), $p(\text{Egyptian cat}) = 0.314$. Solution: `i, j = divmod(s.argmax().item(), 49)`, `Fc[i] = Fd[j]`, `edits.append((i, j))`.
</details>

In [ ]:
def goyal(Fq, Fd):
    c, cd = head(Fq).argmax().item(), head(Fd).argmax().item()
    Fc, edits = Fq.clone(), []
    while head(Fc).argmax().item() != cd:
        # logits after replacing query cell i by distractor cell j: the mean is linear, so we compute all 49 x 49 pairs at once
        L = head(Fc)[None, None, :] + ((Fd[None, :, :] - Fc[:, None, :]) / 49) @ Wfc.T
        s = L.log_softmax(-1)[..., cd]             # log-probability of the distractor's class
        for i, j in edits: s[i] = -np.inf; s[:, j] = -np.inf   # P is a permutation: each cell is used once
        # Task 4: pick the pair (i, j) with the largest s, replace query cell i by distractor cell j and record the pair in edits
        ...  # Your code here: the pair (i, j) with the largest s, the replacement Fc[i] = Fd[j], the record in edits
    return CATS[c], CATS[cd], edits, head(Fc).softmax(0)[cd].item()

q, d = "cat.jpg", "cat_piran.jpg"
c, cd, edits, pcd = goyal(F[q], F[d])
print("%s -> %s: %d edits of 49, pairs (query cell, distractor cell) %s, p(%s) = %.3f" % (c, cd, len(edits), edits, cd, pcd))

In [ ]:
def show_cells(ax, img, idx, color, title):
    im = TF.center_crop(TF.resize(img, prep.resize_size[0]), prep.crop_size[0])   # exactly the frame the model sees
    ax.imshow(im)
    for k in idx:
        r, cc = divmod(k, 7)
        ax.add_patch(plt.Rectangle((cc * 32, r * 32), 32, 32, fill=False, ec=color, lw=3))
    ax.set_title(title, fontsize=10); ax.axis("off")

fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
show_cells(ax[0], imgs[q], [i for i, _ in edits], "#eb6834", f"query: {c}, replaced cell")
show_cells(ax[1], imgs[d], [j for _, j in edits], "#1baf7a", f"distractor: {cd}, source cell")
plt.tight_layout(); plt.show()

On the left the figure shows the query frame as the model sees it after cropping, with the replaced cells in orange frames; on the right the distractor frame, with the cells the features were taken from in green frames. It was enough for the model to see the gray cat's face in place of two cells of the table to the left of the striped cat: what was replaced is not the cat but the background next to it.

Let us check how many replacements other pairs need and how much better the greedy choice is than a random order of pairs.

In [ ]:
def random_edits(Fq, Fd, seed):                    # the same replacement, but in a random order of pairs
    g = np.random.default_rng(seed); cd = head(Fd).argmax().item()
    Fc = Fq.clone()
    for n, (i, j) in enumerate(zip(g.permutation(49), g.permutation(49)), 1):
        Fc[i] = Fd[j]
        if head(Fc).argmax().item() == cd: return n
    return 49

rows = []
for q_, d_ in [("cat.jpg", "cat_piran.jpg"), ("cat_piran.jpg", "cat.jpg"), ("geranimo-ADUi-P4n-Jwds-unsplash.jpg", "piglet.jpg"),
               ("cat.jpg", "cat_and_dog.jpg")]:
    c_, cd_, ed_, p_ = goyal(F[q_], F[d_])
    rnd = [random_edits(F[q_], F[d_], s) for s in range(20)]
    rows.append({"query": f"{q_} ({c_})", "distractor": f"{d_} ({cd_})", "greedy edits": len(ed_),
                 "random edits, median of 20": float(np.median(rnd)), "p of target class": round(p_, 3)})
pd.DataFrame(rows)

In the solved copy the greedy search gives 2 replacements for the pair “striped cat → Egyptian cat”, 7 for “zebra → domestic pig” and 14 for “striped cat → pug”, while a random order needs 6, 19.5 and 43.5 replacements in the median. The reverse pair “Egyptian cat → striped cat” is not symmetric to the direct one: the greedy search replaces 15 cells of 49, a random order 40.5 in the median. An edit covering a third of the frame is no longer a small sparse edit, and it hardly reads as an explanation. The method has a second weakness: the answer depends on the chosen distractor, and the distractor is chosen by hand.

## 5. Time series: Native-Guide

Native-Guide (Delaney et al., 2021) rewrites one segment of a series with the values of the closest series of the other class (step “Time series: Native-Guide”):

$$x^{CF} = (1 - m)\odot x + m\odot \mathrm{NUN},\qquad \min|m|\ \ \text{subject to}\ \ g(x^{CF}) = c'.$$

The steps: find the NUN (native unlike neighbor), the closest training series of the target class $c'$; find the window with the largest sum of the classifier's CAM on the NUN; grow the window length until the prediction changes.

The GunPoint dataset (Ratanamahatana and Keogh) from the UCR Time Series Classification Archive (Dau et al., 2018, [arXiv:1810.07758](https://arxiv.org/abs/1810.07758)); the notebook downloads it from the archive's site [timeseriesclassification.com](https://www.timeseriesclassification.com/description.php?Dataset=GunPoint) and does not copy it to the course `data` folder: the archive states no terms that allow redistributing the data, it only asks to cite the paper. The dataset: the coordinate of an actor's hand over time, 150 samples; in class Gun the actor draws a replica gun from a holster, aims and puts it back, in class Point the actor just points a finger. 50 series for training, 150 for testing. CAM needs a classifier with global averaging, so we take the FCN of Wang et al. (2017): three convolutions, averaging over time and a linear layer, 300 epochs of Adam on the whole training set. The classifier of Delaney et al. is the FCN as implemented by Fawaz et al.: 128, 256 and 128 filters with kernels 8, 5 and 3 and 2000 epochs of Adam with step 0.001. Our FCN is four times narrower, with an odd first kernel of 7 so that the convolution keeps the length of the series, and trains for 300 epochs with step 0.01. Native-Guide itself does not change: it needs any classifier that has a CAM.

In [ ]:
# GunPoint from the UCR/UEA archive site (Dau et al., 2018): the archive states no redistribution terms, so the files are not in the course data/
raw = urllib.request.urlopen("https://www.timeseriesclassification.com/aeon-toolkit/GunPoint.zip", timeout=60).read()
zf = zipfile.ZipFile(io.BytesIO(raw))
tr = np.loadtxt(io.BytesIO(zf.read("GunPoint_TRAIN.txt"))); te = np.loadtxt(io.BytesIO(zf.read("GunPoint_TEST.txt")))
Xtr, ytr = tr[:, 1:].astype(np.float32), (tr[:, 0] - 1).astype(int)    # classes 0 = Gun, 1 = Point
Xte, yte = te[:, 1:].astype(np.float32), (te[:, 0] - 1).astype(int)
print("train", Xtr.shape, "test", Xte.shape, "classes in train", np.bincount(ytr))

class FCN(nn.Module):                              # FCN of Wang et al.: three convolutions, global averaging, a linear layer
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(nn.Conv1d(1, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(),
                               nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
                               nn.Conv1d(64, 32, 3, padding=1), nn.BatchNorm1d(32), nn.ReLU())
        self.fc = nn.Linear(32, 2)
    def forward(self, x): return self.fc(self.f(x[:, None]).mean(-1))
    def cam(self, x, c): return torch.einsum("k,bkt->bt", self.fc.weight[c], self.f(x[:, None]))   # CAM along time

seed_all(0)
fcn = FCN().to(device)
opt = torch.optim.Adam(fcn.parameters(), 1e-2)
Xt, yt = torch.tensor(Xtr, device=device), torch.tensor(ytr, device=device)
t0 = time.time()
for epoch in range(300):
    fcn.train(); loss = nn.functional.cross_entropy(fcn(Xt), yt)
    opt.zero_grad(); loss.backward(); opt.step()
fcn.eval()
def ts_probs(X):
    with torch.no_grad(): return fcn(torch.tensor(np.asarray(X, np.float32), device=device)).softmax(1).cpu().numpy()
print("FCN in %.0f s, test accuracy %.3f" % (time.time() - t0, (ts_probs(Xte).argmax(1) == yte).mean()))

**Task 5.** Complete the NUN search in `native_guide`: among the series `pool` (training series of the target class that the model also assigns to it) take the one closest to `x` in Euclidean distance.

*Self-check question:* which segment had to be rewritten in test series 1, and what share of the series does it cover?

<details><summary>Check yourself</summary>

In the solved copy series 1 of class Point became a series of class Gun after replacing the segment $[81, 108)$, 27 samples of 150, that is 18% of the series. The segment falls on the lowering of the hand, where in class Gun the hand puts the gun back into the holster. The numbers depend on training the FCN. Solution: `nun = pool[np.argmin(((pool - x)**2).sum(1))]`.
</details>

In [ ]:
def native_guide(x):
    c = ts_probs([x])[0].argmax(); cp = 1 - c
    pool = Xtr[(ytr == cp) & (ts_probs(Xtr).argmax(1) == cp)]   # training series of the target class that the model also assigns to it
    # Task 5: NUN is the series in pool closest to x in Euclidean distance
    nun = ...  # Your code here: the series in pool closest to x
    with torch.no_grad(): w = fcn.cam(torch.tensor(nun[None], device=device), cp)[0].cpu().numpy()
    for L in range(1, len(x) + 1):                 # grow the segment length until the prediction changes
        s = int(np.argmax(np.convolve(w, np.ones(L), "valid")))   # window of length L with the largest sum of the NUN's CAM
        xc = x.copy(); xc[s:s + L] = nun[s:s + L]
        if ts_probs([xc])[0, cp] > 0.5: return xc, nun, s, L, c

x = Xte[1]
xc, nun, s, L, c = native_guide(x)
print("series 1: class %d -> %d, rewritten segment [%d, %d), length %d of %d" % (c, 1 - c, s, s + L, L, len(x)))
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(x, color="black", lw=1.5, label="original series")
ax.plot(nun, color="#1baf7a", lw=1, ls="--", label="NUN, the closest series of the other class")
ax.plot(np.arange(s, s + L), xc[s:s + L], color="#eb6834", lw=3, label="rewritten segment")
ax.axvspan(s, s + L, color="#eb6834", alpha=0.1)
ax.set_xlabel("time, samples"); ax.set_ylabel("hand coordinate"); ax.legend(fontsize=8)
ax.set_title("Native-Guide on GunPoint"); plt.show()

In the figure the original series is black, the NUN is the green dashed line, and the rewritten segment is orange; the rest of the series is untouched, and the inserted piece comes from a real series, so the signal stays smooth.

Let us run Native-Guide on all 150 test series and check whether the choice of place by CAM matters: move a segment of the same length from the same NUN to 20 random places of the series.

In [ ]:
def random_segment(x, nun, L, cp, g):              # the same neighbor and length, but the segment at a random place
    s = g.integers(0, len(x) - L + 1); xc = x.copy(); xc[s:s + L] = nun[s:s + L]
    return ts_probs([xc])[0, cp] > 0.5

g = np.random.default_rng(0); rows = []
for i in range(len(Xte)):
    xc, nun, s, L, c = native_guide(Xte[i])
    rows.append({"series": i, "class": c, "length": L, "share of series": L / Xte.shape[1],
                 "random place of the same length": np.mean([random_segment(Xte[i], nun, L, 1 - c, g) for _ in range(20)])})
ng = pd.DataFrame(rows)
print("median length of the rewritten segment: %d of %d samples" % (ng["length"].median(), Xte.shape[1]))
print("median length by original class:", ng.groupby("class")["length"].median().to_dict())
print("share of series where 20%% of the length was enough: %.2f" % (ng["share of series"] <= 0.2).mean())
print("the same segment at a random place changes the prediction in a share of %.2f of cases" % ng["random place of the same length"].mean())
ng.groupby("class")["length"].describe()

In the solved copy the median length of the rewritten segment is 38 samples of 150, and only in 39% of the series was 20% of the length enough. The method is asymmetric: to turn a Point series into a Gun series a short segment on the lowering of the hand is enough, while the reverse transition rewrites more than half of the series, because the FCN's evidence for Gun covers most of the recording. The place chosen by CAM matters: the same segment from the same NUN at a random place changes the prediction in 28% of cases. Native-Guide relies both on CAM and on a neighbor from the sample; if the closest series of the other class is far from the query, the segment grows long and the explanation stops being sparse.

## 6. Why DiME stays in the theory

DiME (Jeanneret, Simon, Jurie, 2022) noises an image with diffusion and on the reverse pass pushes it toward the target class with the classifier's gradient. This needs a diffusion model trained on the same domain as the classifier (CelebA faces and ImageNet in the paper) and hundreds of denoising steps with the classifier's gradient per image. There is no ready model of this scale that runs in free Colab in minutes, so the method is covered in the lesson and not computed in the practice; the same holds for its successors ACE and FastDiME from the step “Conclusion: one statement for all modalities”.

## **Conclusions**

In this practice we checked on three modalities that the counterfactual statement changes only the edit operation and the plausibility model:
- in text, minimality is measured by the word-level Levenshtein distance; MiCE finds an edit more often (10 reviews of 12), because the gradient points at the place and the editor writes for the contrast label, but long edits lose fluency;
- Polyjuice does not look at the classifier: one-word edits are short and fluent, a small share of them changes the prediction, and among them there are edits that change the prediction without changing the meaning, that is, they find a weakness of the model;
- region replacements give a short and clear explanation when the classes are close and the evidence sits in a small area, and degenerate into replacing the whole frame when the evidence for the class is spread over it; in all four pairs the greedy choice needs fewer replacements than the median of a random order;
- Native-Guide rewrites a segment of a real neighbor and keeps the series smooth, and the segment length depends on the direction of the transition and on how close the neighbor is; the place chosen by CAM matters more than a random place of the same length;
- DiME and its successors need a diffusion model and stay in the theory of the lesson.